# Development Set

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Create development set given the manual labeled dataset

In [7]:
import pandas as pd

In [4]:
dev_path = "/content/drive/MyDrive/Comment_Annotations.xlsx"

In [45]:
df_dev = pd.read_excel(dev_path)

In [46]:
df_dev.columns

Index(['ID', 'Date', 'Org. type', 'Org. name', 'Country',
       'Company name confidential', 'Attachment', 'Privacy statement',
       'General Comments', 'Answer to specific info request 2',
       'Answer to specific info request 3',
       'Answer to specific info request 4',
       'Answer to specific info request 5',
       'Answer to specific info request 6',
       'Answer to specific info request 7',
       'Answer to specific info request 8',
       'Answer to specific info request 9',
       'Answer to specific info request 10', 'Source File', 'language',
       'General Comments_translated',
       'Answer to specific info request 1_translated',
       'Answer to specific info request 2_translated',
       'Answer to specific info request 3_translated',
       'Answer to specific info request 4_translated',
       'Answer to specific info request 5_translated',
       'Answer to specific info request 6_translated',
       'Answer to specific info request 7_translated',
    

In [47]:
df_dev = df_dev[["ID","Combined Comments", "Stance"]]

In [48]:
df_dev.shape

(250, 3)

In [1]:
print(df_dev["Stance"].value_counts())

NameError: name 'df_dev' is not defined

Out of those already manually labaled 30 for each class are randomly sampled to form the development dataset.


In [50]:

df_sampled = pd.concat([
    df_dev[df_dev["Stance"] == 1.0].sample(n=30, random_state=42),
    df_dev[df_dev["Stance"] == 2.0].sample(n=30, random_state=42),
    df_dev[df_dev["Stance"] == 3.0]
])


df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)


print(df_sampled["Stance"].value_counts())
print(df_sampled.shape)


Stance
2.0    30
1.0    30
3.0    30
Name: count, dtype: int64
(90, 3)


In [51]:

output_path = "/content/drive/MyDrive/development_set.xlsx"


df_sampled.to_excel(output_path, index=False)


print(f"File saved to: {output_path}")


File saved to: /content/drive/MyDrive/development_set.xlsx


# Zero-shot stance detection

Partly inspired by paper  Cruickshank et al. 2025

In [3]:
API_ENDPOINT = "..."
API_KEY = "..."

In [1]:
import openai

In [5]:
client = openai.OpenAI(
    api_key= API_KEY,
    base_url= API_ENDPOINT
)

## Model 1: Task-only stance

In [10]:
input_path = "/content/drive/MyDrive/development_set.xlsx"
output_path = "/content/drive/MyDrive/development_output.xlsx"
checkpoint_interval = 5

In [9]:
import pandas as pd
import openai
import time
import os
from tqdm import tqdm


This ensures that if the code stop running for some reason I can go back to the last checkpoint

In [30]:
if os.path.exists(output_path):
    df = pd.read_excel(output_path)
    print("Resuming from previous checkpoint")
else:
    df = pd.read_excel(input_path)
    if "output_1" not in df.columns:
        df["output_1"] = ""

In [31]:
def task_only(comment):
  return f"""
Classify the stance of the following comment toward the proposed PFAS restriction policy:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

This function is the same for all prompt

In [15]:
def get_response(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Error: {e}")
        return "error"

In [33]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_1"]:
        stance = get_response(task_only(comment))
        df.at[idx, "output_1"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:02<03:35,  2.43s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:36,  1.78s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:06<00:01,  1.35s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:08<00:00,  1.42s/it]

Classification completed and saved.


## Model 2: Task Definition

In [73]:
input_path = "/content/drive/MyDrive/development_set.xlsx"
output_path = "/content/drive/MyDrive/development_output.xlsx"
checkpoint_interval = 5

In [61]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_2"

if current_output_col not in df.columns:
    df[current_output_col] = ""


Resuming from previous checkpoint


In [63]:
def task_def(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

Classify the stance of the following comment toward the proposed PFAS restriction policy:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""


In [64]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_2"]:
        stance = get_response(task_def(comment))
        df.at[idx, "output_2"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:52,  1.94s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:19,  1.58s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:05<00:01,  1.36s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:07<00:00,  1.41s/it]

 Classification completed and saved.


## Model 3: Context Analyze



In [92]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_3"

if current_output_col not in df.columns:
    df[current_output_col] = ""


Resuming from previous checkpoint


In [93]:
def task_context(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

The following is a public consultation comment related to the proposed restriction of PFAS substances.

Classify the stance expressed in the comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""


In [94]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_3"]:
        stance = get_response(task_context(comment))
        df.at[idx, "output_3"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:46,  1.87s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:18,  1.57s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:08<00:01,  1.61s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:09<00:00,  1.44s/it]

 Classification completed and saved.


## Model 3.1: Role Context Analyze

Da considerare quale tenere dei due

In [95]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_3.1"

if current_output_col not in df.columns:
    df[current_output_col] = ""


Resuming from previous checkpoint


In [96]:
def task_context_role(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Classify the stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""


In [97]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_3.1"]:
        stance = get_response(task_context_role(comment))
        df.at[idx, "output_3.1"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:44,  1.85s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:15,  1.54s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:07<00:01,  1.37s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:09<00:00,  1.43s/it]

 Classification completed and saved.


## Model 4: Context Question

In [98]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_4"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [99]:
def task_context_question(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

The following is a public consultation comment related to the proposed restriction of PFAS substances.

Question: What is the stance expressed in the comment toward the PFAS restriction proposal?

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [100]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_4"]:
        stance = get_response(task_context_question(comment))
        df.at[idx, "output_4"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:44,  1.85s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:13,  1.52s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:09<00:01,  1.37s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:12<00:00,  1.47s/it]

 Classification completed and saved.


## Model 4.1: Role Context Question

In [101]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_4.1"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [102]:
def task_context_question_role(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Question: What is the stance expressed in the comment toward the PFAS restriction proposal?

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [103]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_4.1"]:
        stance = get_response(task_context_question_role(comment))
        df.at[idx, "output_4.1"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:44,  1.84s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:13,  1.52s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:12<00:01,  1.37s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:15<00:00,  1.50s/it]

 Classification completed and saved.


## Model 5: Chain of Thoughts

In [104]:

df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_5"
cot_raw_col = "cot_raw"

# Ensure both columns exist
for col in [current_output_col, cot_raw_col]:
    if col not in df.columns:
        df[col] = ""

Resuming from previous checkpoint


In [106]:
def task_cot_explanation(comment):
    return f"""

Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Think step-by-step and explain the stance (in favor, against, or neutral) of the comment toward the PFAS restriction policy.

Comment:
\"{comment}\"

Explanation:
"""


In [107]:
def task_cot_answer(comment, explanation):
    return f"""

Given your explanation: {explanation}

Determine the final stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""


In [108]:
# MAIN LOOP for Chain-of-Thought
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STEP 1: Generate explanation (cot_raw)
    if not row[cot_raw_col]:
        explanation = get_response(task_cot_explanation(comment))
        df.at[idx, cot_raw_col] = explanation
        print(f"[{idx}] CoT Explanation → {explanation}")
        time.sleep(1)
    else:
        explanation = row[cot_raw_col]

    # STEP 2: Final stance prediction based on explanation (output_5)
    if not row[current_output_col]:
        stance = get_response(task_cot_answer(comment, explanation))
        df.at[idx, current_output_col] = stance
        print(f"[{idx}] Final Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f" Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Chain-of-Thought classification completed and saved.")


Classifying comments - Model 5:   0%|          | 0/90 [00:00<?, ?it/s]

[0] CoT Explanation → The comment expresses a complex stance toward the proposed restriction of PFAS substances. Let's break it down step-by-step:

1. **Support for Minimizing Hazardous Substances**: The comment begins with a general statement that everyone should support efforts to minimize the impact of hazardous substances on human health and the environment. This indicates a recognition of the importance of addressing hazardous substances, which could be seen as a neutral or slightly positive stance.

2. **Critique of the Proposal**: The author then critiques the current restriction project, stating that it "goes far beyond" what is necessary. This indicates a clear opposition to the specific proposal as it stands, suggesting that the author believes the proposal is excessive or misguided.

3. **Concerns About the Proposal's Basis**: The comment raises concerns about the justification for the total ban on all PFAS, arguing that many PFAS do not meet the criteria for being harmful (

Classifying comments - Model 5:   1%|          | 1/90 [00:06<09:13,  6.22s/it]

💾 Checkpoint saved at row 0
[1] CoT Explanation → To analyze the stance of the comment regarding the proposed restriction of PFAS substances, we can break it down step-by-step:

1. **Identify the Subject**: The comment discusses PFAS (per- and polyfluoroalkyl substances) and their impact in Sweden.

2. **Assess the Language Used**: The phrase "has already caused great damage" indicates a negative impact associated with PFAS. This suggests that the commenter views PFAS in a harmful light.

3. **Determine the Action Suggested**: The comment states, "it is necessary to stop it as soon as possible." This indicates a strong urgency and a call for action against PFAS.

4. **Evaluate the Overall Sentiment**: The combination of acknowledging the damage caused by PFAS and the urgent call to stop it implies a clear opposition to the continued use or presence of PFAS.

5. **Conclude the Stance**: Given that the comment expresses a negative view of PFAS and advocates for immediate action to restri

Classifying comments - Model 5:   2%|▏         | 2/90 [00:10<07:17,  4.97s/it]

...

[88] CoT Explanation → To analyze the stance of the comment toward the proposed restriction of PFAS substances, we can break it down step-by-step:

1. **Identification of Key Phrases**: The comment begins with "I think the proposed change is good as it is," which indicates a positive view toward the proposed restriction. The phrase "I hope it will be adopted without being watered down" further emphasizes the desire for the policy to remain strong and effective.

2. **Emphasis on Importance**: The comment states, "It is very important that we stop the emissions of long-lasting chemicals." This highlights the urgency and significance the commenter places on addressing the issue of PFAS emissions, suggesting a strong support for the restriction.

3. **Comparison of Priorities**: The phrase "this should weigh much heavier than the potential impact on different products in terms of functionality" indicates that the commenter believes the health and environmental benefits of restricting PFAS

Classifying comments - Model 5:  99%|█████████▉| 89/90 [09:19<00:06,  6.55s/it]

[89] CoT Explanation → The comment provided expresses a clear stance against the proposed restriction of PFAS substances, particularly in relation to fluorine-based polymer and elastomer products used in water treatment and other critical infrastructure. Here’s a step-by-step breakdown of the reasoning:

1. **Advocacy for Specific Products**: The Association of Membrane Separation Technology of Japan (AMST) explicitly states that they "strongly advocate" for the availability of fluorine-based products for water treatment plants. This indicates a preference for these products rather than a neutral or supportive stance toward the proposed restrictions.

2. **Argument Against Generalization**: The comment argues that regulations should not group all PFAS materials together, suggesting that the AMST believes that not all PFAS substances pose a risk to human health. This implies a disagreement with the broad application of the proposed regulations.

3. **Emphasis on Essentiality**: The comm

Classifying comments - Model 5: 100%|██████████| 90/90 [09:25<00:00,  6.28s/it]

✅ Chain-of-Thought classification completed and saved.


# One-shot stance detection

To perform one-shot stance detection three models are run. The first two presents two different sets of examples to the model in the prompt. Every set is composed of one example per class (in favor, against, neutral).

The two sets differ in terms of stance clarity as follows:

-The first set contains comments in which the stance is clearly and explicitly stated.

-The second set contains comments in which the stance is expressed implicilty.

The third model include CoT to the implict set


## Model 6: Explicit stance

In [11]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_6"

if current_output_col not in df.columns:
    df[current_output_col] = ""


Resuming from previous checkpoint


In [12]:
comment_1 = """I support the limitation proposal as it is and hope that it will be adopted as it looks today. PFAS have proven to be toxic and persistent chemicals that negatively affect both human health and the environment, and I therefore only see it as a positive thing that their use is being stopped, especially in products where they are not essential. For example, in makeup, electronics, or in clothes! I believe that it is more beneficial for humanity in the long run if the quality of products potentially decreases slightly (e.g. a raincoat not repelling water as well) than if we become more exposed to higher concentrations of persistent chemicals like PFAS."""
comment_2 = """We acknowledge  the  necessity  of  regulating  PFAS,  but  the  current  restriction proposal is too broad and too impacting to be relevant.  Fluoropolymers (including fluoroelastomers)  should  be  out  of  scope  of  this  restriction,  and  the  “repair  as produced” principle should be respected for all existing vehicles. Fluoropolymers (including fluoroelastomers) were quest their removal from the scope of the restriction. Concerning the manufacturing phase, the risks of PFAS emissions to the environment can be controlled with alternative Risk Management Options. Concerning the use phase, they are considered non- toxic, non-bioaccumulative, non-mobile and as such, are classed as polymers of low concern. Concerning the end-of-life phase, incineration of fluoropolymers does not contribute to environmental PFAS emissions and is a safe method of disposal. 7296       The PFAS REACH restriction is expected to have a major impact on the automotive industry. The automotive industry is a major downstream user of many PFAS, including fluoropolymers, fluorinated gases, and short-chain PFAS. Fluoropolymers are used for several key technical components, such as gaskets, hoses, joints, O-rings, seals, cords, cables, or sleeves. The current proposal does not acknowledge any derogations for such uses, whereas alternatives are not readily available and do not share sufficient properties to be qualified. In the context of this derogation, the automotive industry wishes to express its great concern if the implementation of the restriction were to continue as is and proposes an alternative implementation approach that integrates the technical and economic constraints on the one hand and preserves the objectives of electromobility on the other."""
comment_3 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries. """

In [13]:
def one_explicit(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: Against

Example 3:
Comment 3: {comment_3}
Stance 3: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [16]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_6"]:
        stance = get_response(one_explicit(comment))
        df.at[idx, "output_6"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:02<03:11,  2.16s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:46,  1.90s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:35<00:01,  1.66s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:36<00:00,  1.74s/it]

 Classification completed and saved.


## Model 7: Implicit stance

In [17]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_7"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [18]:
comment_1 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_2 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_3 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""

In [19]:
def one_implicit(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: Against

Example 3:
Comment 3: {comment_3}
Stance 3: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [20]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_7"]:
        stance = get_response(one_implicit(comment))
        df.at[idx, "output_7"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:18,  1.56s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:14,  1.52s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:24<00:01,  1.58s/it]

[89] Stance → In favor


Classifying comments: 100%|██████████| 90/90 [02:25<00:00,  1.62s/it]

 Classification completed and saved.


## Model 8: CoT Explanation

In [21]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_8"
cot_raw_col = "one_cot"

# Ensure both columns exist
for col in [current_output_col, cot_raw_col]:
    if col not in df.columns:
        df[col] = ""

Resuming from previous checkpoint


In [22]:
comment_1 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_2 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_3 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""

In [23]:
explanation_1 = """The commenter emphasizes the need for clean drinking water from a medical perspective. In particular, they highlight that economic interests should not be prioritized over human health. While PFAS are not explicitly mentioned, their negative effects on human health due to contamination of drinking water are well known. For this reason, the commenter appears to support the restriction of PFAS use, aligning with the goals of the PFAS policy proposal."""
explanation_2 = """The commenter expresses support for the statements made by FCJ and JFIA which find the proposed PFAS restriction excessive and that affirm that fluoropolymers should be exempted from it. The commenter also provides context on the use of PFAS-related substances such as FKM, PTFE and PCTFE. Their usage is essential for vacuum sealing, tubing, and gaskets in equipment used across several sectors such as pharmaceuticals and electronics. The commenters highlight the absence of alternatives with equivalent performance. It also states that information about annual tonnage and emissions are not available. For these reasons the commenter appears to oppose the PFAS restriction policy."""
explanation_3 = """The commenter indicates their intent to provide detailed information on the use of PFAS in the chemical industry, specifically in the chemicals synthesis. However, the relevant content is shared in two attachments of which one is confidential. Without access to the attachments it is hard to classify the comment position toward the PFAS restriction. For this reason the comment appears to be neutral with respect to the policy."""

In [24]:
def one_shot_cot_explanation(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Explanation 1: {explanation_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Explanation 2: {explanation_2}
Stance 2: Against

Example 3:
Comment 3: {comment_3}
Explanantion 3: {explanation_3}
Stance 3: Neutral

Now think step-by-step and explain the stance (in favor, against, or neutral) of the comment toward the PFAS restriction policy.

Comment:
\"{comment}\"

Explanation:
"""


In [25]:
def one_shot_cot_answer(comment, explanation):
    return f"""
Given your explanation: {explanation}

Determine the final stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""


In [26]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STEP 1: Generate explanation (one_cot)
    if not row[cot_raw_col]:
        explanation = get_response(one_shot_cot_explanation(comment))
        df.at[idx, cot_raw_col] = explanation
        print(f"[{idx}]  Explanation → {explanation}")
        time.sleep(1)
    else:
        explanation = row[cot_raw_col]

    # STEP 2: Final stance prediction (output_8)
    if not row[current_output_col]:
        stance = get_response(one_shot_cot_answer(comment, explanation))
        df.at[idx, current_output_col] = stance
        print(f"[{idx}]  Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f" Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")


Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0]  Explanation → The commenter begins by expressing general support for efforts to minimize the impact of hazardous substances on human health and the environment. However, they quickly pivot to criticize the current restriction project on PFAS, stating that it goes too far and should be rejected in its current form. They argue that the proposal to ban all PFAS is excessive and lacks sufficient justification, particularly in terms of demonstrating an unacceptable risk to human health or the environment.

The commenter provides a detailed critique of the proposal, highlighting that many PFAS are not bioavailable, mobile, or toxic, and therefore do not meet the criteria for a lawful restriction under REACH regulations. They emphasize the importance of fluoropolymers, such as PTFE, in various industrial applications, arguing that banning these substances would significantly hinder safe chemical handling and production processes. The commenter warns that such a ban could lead to de-indus

Classifying comments:   1%|          | 1/90 [00:06<09:28,  6.39s/it]

 Checkpoint saved at row 0
[1]  Explanation → The commenter expresses a clear concern regarding the impact of PFAS substances, specifically mentioning that they have already caused significant damage in Sweden. The use of the phrase "it is necessary to stop it as soon as possible" indicates a strong urgency and a call for action against PFAS. This statement reflects a position that aligns with the goals of the PFAS restriction policy, which aims to mitigate the harmful effects of these substances.

Given the explicit mention of the damage caused by PFAS and the urgent call to stop their use, the comment clearly supports the restriction of PFAS substances.

Stance: In favor
[1]  Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:11<08:37,  5.88s/it]

...

[88]  Explanation → The commenter expresses a clear opinion regarding the proposed change, indicating that they believe it is a positive step and should be adopted in its current form without modifications. They emphasize the importance of stopping emissions of long-lasting chemicals, which aligns with the goals of the PFAS restriction policy. The mention of prioritizing environmental health over potential impacts on product functionality further reinforces their support for the restriction. 

Given these points, the stance of the comment toward the PFAS restriction policy is:

Stance: In favor
[88]  Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [10:05<00:07,  7.23s/it]

[89]  Explanation → The commenter, representing the Association of Membrane Separation Technology of Japan (AMST), expresses a strong position against the proposed PFAS regulation by the EU. They advocate for the continued use of fluorine-based polymer and elastomer products in critical applications such as water treatment plants, emphasizing their essential role in providing safe drinking water and facilitating wastewater recycling. 

The commenter argues that not all PFAS materials should be grouped together under the same regulations, suggesting that only those substances that pose a risk to human health should be restricted. They highlight the importance of these fluorine-based products in maintaining public health, protecting the environment, and ensuring a stable water supply. Furthermore, they point out the lack of viable alternatives to these products, indicating that replacing them would impose significant burdens on both companies and the general public, potentially leading t

Classifying comments: 100%|██████████| 90/90 [10:13<00:00,  6.82s/it]

 Classification completed and saved.


# Few-shot stance detection

To perform few-shot stance detection three models are run. For each of them three different sets of examples are presented to the model in the prompt.

The first two models aim to test whether the examples' order affect model's performance. Both models take the same set of examples which is composed of three example per class (in favor, against, neutral).

The first two models' prompts differ as follows:

-The first lists the examples grouping them by stance (i.e. first all the in favor, then those against and finally the neutral ones).

-The second prompt alternate the order of the examples.

The third model aims to test whether a class unbalanced example set might lead to output bias. To do so the prompt is provided with an example set of 5 in favor, 2 against and 2 neutral.



## Model 9: Grouped stances

In [27]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_9"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [28]:
comment_1 = """I support the limitation proposal as it is and hope that it will be adopted as it looks today. PFAS have proven to be toxic and persistent chemicals that negatively affect both human health and the environment, and I therefore only see it as a positive thing that their use is being stopped, especially in products where they are not essential. For example, in makeup, electronics, or in clothes! I believe that it is more beneficial for humanity in the long run if the quality of products potentially decreases slightly (e.g. a raincoat not repelling water as well) than if we become more exposed to higher concentrations of persistent chemicals like PFAS."""
comment_2 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_3 = """Over the last years, perfluorocarbons and perfluoropolyethers have been frequently used as dielectric fluids for immersion cooling applications. The use of such liquids entails a high environmental impact as well as severe health risks during operation, maintenance and handling of the immersed equipment. Recently, new and more sustainable dielectric liquids – able to efficiently operate these systems - have entered the market, resulting in no need for per- and polyfluoroalkyl substances anymore. 9165 Cleaning and heat transfer: engineered fluids. Electronics: heat transfer fluid for (single-phase) immersion cooling (in EV, IT equipment). 9165       PFAS alternatives for (single-phase) immersion cooling applications. Recently, sustainable hydrocarbon (e.g. DC COOLING BioLife 4 from TotalEnergies) and ester (e.g. NatureCool2000 from Cargill, Mivolt DF7 from M&I Materials) based dielectric liquids have entered the immersion cooling market."""

comment_4 = """We acknowledge  the  necessity  of  regulating  PFAS,  but  the  current  restriction proposal is too broad and too impacting to be relevant.  Fluoropolymers (including fluoroelastomers)  should  be  out  of  scope  of  this  restriction,  and  the  “repair  as produced” principle should be respected for all existing vehicles. Fluoropolymers (including fluoroelastomers) were quest their removal from the scope of the restriction. Concerning the manufacturing phase, the risks of PFAS emissions to the environment can be controlled with alternative Risk Management Options. Concerning the use phase, they are considered non- toxic, non-bioaccumulative, non-mobile and as such, are classed as polymers of low concern. Concerning the end-of-life phase, incineration of fluoropolymers does not contribute to environmental PFAS emissions and is a safe method of disposal. 7296       The PFAS REACH restriction is expected to have a major impact on the automotive industry. The automotive industry is a major downstream user of many PFAS, including fluoropolymers, fluorinated gases, and short-chain PFAS. Fluoropolymers are used for several key technical components, such as gaskets, hoses, joints, O-rings, seals, cords, cables, or sleeves. The current proposal does not acknowledge any derogations for such uses, whereas alternatives are not readily available and do not share sufficient properties to be qualified. In the context of this derogation, the automotive industry wishes to express its great concern if the implementation of the restriction were to continue as is and proposes an alternative implementation approach that integrates the technical and economic constraints on the one hand and preserves the objectives of electromobility on the other."""
comment_5 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_6 = """As engineering service provider, planner and constructor for large-scale plant construction, we specify and handle various materials and material groups. Among others, the following semi-finished products and assemblies made of fluoropolymers are used in our range of activities: - Fluoropolymers as sealing materials (flat, O-ring, mechanical seal, etc.) in various equipment/pipelines - Fluoropolymers as lining materials in pumps and pipeline materials - Fluoropolymers as membrane materials in pumps, electrolyzers Additionally, fluoropolymers are indirectly used as coolants or additives in associated processes, which however fall within the scope of the end user/customer. As the broad field of mechanical and plant engineering is not listed in the sectors mentioned in Annex XV (Table 9) of the restriction report, the following areas were selected as the most relevant - Energy sector (Annex E.2.12.) - Sector as a whole - Petroleum and mining (Annex E.2.15.) - Fluoropolymer applications Furthermore, see SECTION III. Non-confidential comments. 9567" "We, as large-scale plant builders, can divide the percentage emission shares for our scope as follows: 0% emissions in the production phase (as we appear as end users) 0% emissions in the usage phase 100% emissions in the end-of-life phase (this is outside our scope and not considered) 9567"    "See SECTION III. Non-confidential comments"""

comment_7 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries. """
comment_8 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""
comment_9 = """Integrated DNA Technologies, a Biotechnology company, develops, manufactures, and markets nucleic acid (DNA and RNA) products that support the life sciences industry. IDT supplies synthetic biology products used in genomics research, diagnostics, specialized reagents, consumables, and medical device products. Our customers use our products in applications ranging from fundamental biological and genomics research to making lifesaving vaccines, biologic drugs, and novel cell and gene therapies. We supply the tools and services that help our customers do their work better, faster, and safer. IDT designs, builds, and then deploys proprietary manufacturing equipment into the European Union to manufacture nucleic acid products. We have over 1400 associates across nine countries worldwide including manufacturing sites in Belgium, Singapore, and the USA.  General Comments According to ECHA’s Press Release, the purpose of the consultation is to give anyone with information on PFAS the opportunity to have their say. Of particular interest is information relevant to the risks, socio-economic aspects, and alternative substances. ECHA’s scientific committees for Risk Assessment (RAC) and for Socio-Economic Analysis (SEAC) will use the consultation input to evaluate the proposed restriction and to form an opinion on it.  However, due to the rather high volume of the information that must be collected and analysed with regard to the economic and health impacts of the proposed restriction, we request a 6-month extension to the consultation period, in order to gather detailed data on these points."""

In [29]:
def few_grouped(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:


Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: In favor

Example 3:
Comment 3: {comment_3}
Stance 3: In favor

Example 4:
Comment 4: {comment_4}
Stance 4: Against

Example 5:
Comment 5: {comment_5}
Stance 5: Against

Example 6
Comment 6: {comment_6}
Stance 6: Against

Example 7:
Comment 7: {comment_7}
Stance 7: Neutral

Example 8:
Comment 8: {comment_8}
Stance 8: Neutral

Example 9:
Comment 9: {comment_9}
Stance 9: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:


Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [30]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_9"]:
        stance = get_response(few_grouped(comment))
        df.at[idx, "output_9"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:39,  1.79s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:30,  1.71s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:39<00:01,  1.76s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:41<00:00,  1.79s/it]

 Classification completed and saved.


## Model 10: Alternating stances

In [31]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_10"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [32]:
comment_1 = """I support the limitation proposal as it is and hope that it will be adopted as it looks today. PFAS have proven to be toxic and persistent chemicals that negatively affect both human health and the environment, and I therefore only see it as a positive thing that their use is being stopped, especially in products where they are not essential. For example, in makeup, electronics, or in clothes! I believe that it is more beneficial for humanity in the long run if the quality of products potentially decreases slightly (e.g. a raincoat not repelling water as well) than if we become more exposed to higher concentrations of persistent chemicals like PFAS."""
comment_2 = """We acknowledge  the  necessity  of  regulating  PFAS,  but  the  current  restriction proposal is too broad and too impacting to be relevant.  Fluoropolymers (including fluoroelastomers)  should  be  out  of  scope  of  this  restriction,  and  the  “repair  as produced” principle should be respected for all existing vehicles. Fluoropolymers (including fluoroelastomers) were quest their removal from the scope of the restriction. Concerning the manufacturing phase, the risks of PFAS emissions to the environment can be controlled with alternative Risk Management Options. Concerning the use phase, they are considered non- toxic, non-bioaccumulative, non-mobile and as such, are classed as polymers of low concern. Concerning the end-of-life phase, incineration of fluoropolymers does not contribute to environmental PFAS emissions and is a safe method of disposal. 7296       The PFAS REACH restriction is expected to have a major impact on the automotive industry. The automotive industry is a major downstream user of many PFAS, including fluoropolymers, fluorinated gases, and short-chain PFAS. Fluoropolymers are used for several key technical components, such as gaskets, hoses, joints, O-rings, seals, cords, cables, or sleeves. The current proposal does not acknowledge any derogations for such uses, whereas alternatives are not readily available and do not share sufficient properties to be qualified. In the context of this derogation, the automotive industry wishes to express its great concern if the implementation of the restriction were to continue as is and proposes an alternative implementation approach that integrates the technical and economic constraints on the one hand and preserves the objectives of electromobility on the other."""
comment_3 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries."""

comment_4 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_5 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_6 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""

comment_7 = """Over the last years, perfluorocarbons and perfluoropolyethers have been frequently used as dielectric fluids for immersion cooling applications. The use of such liquids entails a high environmental impact as well as severe health risks during operation, maintenance and handling of the immersed equipment. Recently, new and more sustainable dielectric liquids – able to efficiently operate these systems - have entered the market, resulting in no need for per- and polyfluoroalkyl substances anymore. 9165 Cleaning and heat transfer: engineered fluids. Electronics: heat transfer fluid for (single-phase) immersion cooling (in EV, IT equipment). 9165       PFAS alternatives for (single-phase) immersion cooling applications. Recently, sustainable hydrocarbon (e.g. DC COOLING BioLife 4 from TotalEnergies) and ester (e.g. NatureCool2000 from Cargill, Mivolt DF7 from M&I Materials) based dielectric liquids have entered the immersion cooling market."""
comment_8 = """As engineering service provider, planner and constructor for large-scale plant construction, we specify and handle various materials and material groups. Among others, the following semi-finished products and assemblies made of fluoropolymers are used in our range of activities: - Fluoropolymers as sealing materials (flat, O-ring, mechanical seal, etc.) in various equipment/pipelines - Fluoropolymers as lining materials in pumps and pipeline materials - Fluoropolymers as membrane materials in pumps, electrolyzers Additionally, fluoropolymers are indirectly used as coolants or additives in associated processes, which however fall within the scope of the end user/customer. As the broad field of mechanical and plant engineering is not listed in the sectors mentioned in Annex XV (Table 9) of the restriction report, the following areas were selected as the most relevant - Energy sector (Annex E.2.12.) - Sector as a whole - Petroleum and mining (Annex E.2.15.) - Fluoropolymer applications Furthermore, see SECTION III. Non-confidential comments. 9567" "We, as large-scale plant builders, can divide the percentage emission shares for our scope as follows: 0% emissions in the production phase (as we appear as end users) 0% emissions in the usage phase 100% emissions in the end-of-life phase (this is outside our scope and not considered) 9567"    "See SECTION III. Non-confidential comments"""
comment_9 = """Integrated DNA Technologies, a Biotechnology company, develops, manufactures, and markets nucleic acid (DNA and RNA) products that support the life sciences industry. IDT supplies synthetic biology products used in genomics research, diagnostics, specialized reagents, consumables, and medical device products. Our customers use our products in applications ranging from fundamental biological and genomics research to making lifesaving vaccines, biologic drugs, and novel cell and gene therapies. We supply the tools and services that help our customers do their work better, faster, and safer. IDT designs, builds, and then deploys proprietary manufacturing equipment into the European Union to manufacture nucleic acid products. We have over 1400 associates across nine countries worldwide including manufacturing sites in Belgium, Singapore, and the USA.  General Comments According to ECHA’s Press Release, the purpose of the consultation is to give anyone with information on PFAS the opportunity to have their say. Of particular interest is information relevant to the risks, socio-economic aspects, and alternative substances. ECHA’s scientific committees for Risk Assessment (RAC) and for Socio-Economic Analysis (SEAC) will use the consultation input to evaluate the proposed restriction and to form an opinion on it.  However, due to the rather high volume of the information that must be collected and analysed with regard to the economic and health impacts of the proposed restriction, we request a 6-month extension to the consultation period, in order to gather detailed data on these points."""

In [33]:
def few_altern(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: Against

Example 3:
Comment 3: {comment_3}
Stance 3: Neutral

Example 4:
Comment 4: {comment_4}
Stance 4: In favor

Example 5:
Comment 5: {comment_5}
Stance 5: Against

Example 6
Comment 6: {comment_6}
Stance 6: Neutral

Example 7:
Comment 7: {comment_7}
Stance 7: In favor

Example 8:
Comment 8: {comment_8}
Stance 8: Against

Example 9:
Comment 9: {comment_9}
Stance 9: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:
Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [34]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_10"]:
        stance = get_response(few_altern(comment))
        df.at[idx, "output_10"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:29,  1.68s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:22,  1.62s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:24<00:01,  1.61s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:26<00:00,  1.62s/it]

 Classification completed and saved.


## Model 11: Class-Imbalance stances

## Modell 11.0 : More Against

In [35]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_11.0"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [36]:
comment_1 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_2 = """Over the last years, perfluorocarbons and perfluoropolyethers have been frequently used as dielectric fluids for immersion cooling applications. The use of such liquids entails a high environmental impact as well as severe health risks during operation, maintenance and handling of the immersed equipment. Recently, new and more sustainable dielectric liquids – able to efficiently operate these systems - have entered the market, resulting in no need for per- and polyfluoroalkyl substances anymore. 9165 Cleaning and heat transfer: engineered fluids. Electronics: heat transfer fluid for (single-phase) immersion cooling (in EV, IT equipment). 9165       PFAS alternatives for (single-phase) immersion cooling applications. Recently, sustainable hydrocarbon (e.g. DC COOLING BioLife 4 from TotalEnergies) and ester (e.g. NatureCool2000 from Cargill, Mivolt DF7 from M&I Materials) based dielectric liquids have entered the immersion cooling market."""

comment_3 = """We acknowledge  the  necessity  of  regulating  PFAS,  but  the  current  restriction proposal is too broad and too impacting to be relevant.  Fluoropolymers (including fluoroelastomers)  should  be  out  of  scope  of  this  restriction,  and  the  “repair  as produced” principle should be respected for all existing vehicles. Fluoropolymers (including fluoroelastomers) were quest their removal from the scope of the restriction. Concerning the manufacturing phase, the risks of PFAS emissions to the environment can be controlled with alternative Risk Management Options. Concerning the use phase, they are considered non- toxic, non-bioaccumulative, non-mobile and as such, are classed as polymers of low concern. Concerning the end-of-life phase, incineration of fluoropolymers does not contribute to environmental PFAS emissions and is a safe method of disposal. 7296       The PFAS REACH restriction is expected to have a major impact on the automotive industry. The automotive industry is a major downstream user of many PFAS, including fluoropolymers, fluorinated gases, and short-chain PFAS. Fluoropolymers are used for several key technical components, such as gaskets, hoses, joints, O-rings, seals, cords, cables, or sleeves. The current proposal does not acknowledge any derogations for such uses, whereas alternatives are not readily available and do not share sufficient properties to be qualified. In the context of this derogation, the automotive industry wishes to express its great concern if the implementation of the restriction were to continue as is and proposes an alternative implementation approach that integrates the technical and economic constraints on the one hand and preserves the objectives of electromobility on the other."""
comment_4 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_5 = """As engineering service provider, planner and constructor for large-scale plant construction, we specify and handle various materials and material groups. Among others, the following semi-finished products and assemblies made of fluoropolymers are used in our range of activities: - Fluoropolymers as sealing materials (flat, O-ring, mechanical seal, etc.) in various equipment/pipelines - Fluoropolymers as lining materials in pumps and pipeline materials - Fluoropolymers as membrane materials in pumps, electrolyzers Additionally, fluoropolymers are indirectly used as coolants or additives in associated processes, which however fall within the scope of the end user/customer. As the broad field of mechanical and plant engineering is not listed in the sectors mentioned in Annex XV (Table 9) of the restriction report, the following areas were selected as the most relevant - Energy sector (Annex E.2.12.) - Sector as a whole - Petroleum and mining (Annex E.2.15.) - Fluoropolymer applications Furthermore, see SECTION III. Non-confidential comments. 9567" "We, as large-scale plant builders, can divide the percentage emission shares for our scope as follows: 0% emissions in the production phase (as we appear as end users) 0% emissions in the usage phase 100% emissions in the end-of-life phase (this is outside our scope and not considered) 9567"    "See SECTION III. Non-confidential comments"""
comment_6 = """We welcome the Commission’s proposal to revise the PFAS regulation and support the point of view of the FEA (European Aerosol Federation). Supplementary information is provided in Section V. 8860 Applications of fluorinated gases (Annex E.2.8.), Propellants Propellants : for technical aerosols for applications where non-flammability and high technical performance of spray are required, p 126 ,table 9 8860    Regarding the proposed derogation : propellants for technical aerosols for applications where non-flammability and high technical performance of spray quality are required (p126, table 9) -General info was shared by the FEA. -Supplementary info is provided in section V. 8860  -General information was shared by the FEA. -Supplementary info is provided in Section V"""
comment_7 = """So far, no derogation has been made for waste treatment and recycling technologies. However, these are crucial for enabeling the circular economy and should thus have an exemption. 9576  The bachelor thesis of Emine Sürmeli at Hochschule Rhein Main, summerized different end of life options for Flourpolymers. Also we can provide contact with a company how actually does recycling PTFE. 9576  To our knowledge it is currently not possible to estimate the impact the restriction would have on the recycling process and in how far the proposed concentration limits would be met. As far as we know, currently PFAS are not meassured in waste streams. Thus it is not clear up to which amount PFAS are included in waste streams. Recycler currently do not have any testing option for PFAS. 9576  In the waste treatment and recycling technology sector PFAS, more specifically Fluorpolymers may be used in the following examples. The list is non-exhaustiv: - Air conditioning units in mobile vehicles - Seals e.g. in engines - Cable sheathing in machinery and equipment where it must be fire retardant - Insulations/dampers - In components e.g. hydraulic components, fittings, pumps, bearing bushings, electronics, sensors - In production e.g. as PTFE heating plates for plastic welding - Baffle plates Other applications and/or components may be effected as well. There is no clear overview of the PFAS application in waste treatment and recycling technologies."""

comment_8 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries. """
comment_9 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""


In [37]:
def few_imbalance_against(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: In favor

Example 3:
Comment 3: {comment_3}
Stance 3: Against

Example 4:
Comment 4: {comment_4}
Stance 4: Against

Example 5:
Comment 5: {comment_5}
Stance 5: Against

Example 6
Comment 6: {comment_6}
Stance 6: Against

Example 7:
Comment 7: {comment_7}
Stance 7: Against

Example 8:
Comment 8: {comment_8}
Stance 8: Neutral

Example 9:
Comment 9: {comment_9}
Stance 9: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [38]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_11.0"]:
        stance = get_response(few_imbalance_against(comment))
        df.at[idx, "output_11.0"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:22,  1.61s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:19,  1.58s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:27<00:01,  1.57s/it]

[89] Stance → In favor


Classifying comments: 100%|██████████| 90/90 [02:29<00:00,  1.66s/it]

 Classification completed and saved.


## Model 11.1 : More In favor

In [39]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "output_11.1"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [40]:
comment_1 = """I support the limitation proposal as it is and hope that it will be adopted as it looks today. PFAS have proven to be toxic and persistent chemicals that negatively affect both human health and the environment, and I therefore only see it as a positive thing that their use is being stopped, especially in products where they are not essential. For example, in makeup, electronics, or in clothes! I believe that it is more beneficial for humanity in the long run if the quality of products potentially decreases slightly (e.g. a raincoat not repelling water as well) than if we become more exposed to higher concentrations of persistent chemicals like PFAS."""
comment_2 = """As a medical doctor with a special interest in human fluid balance, the importance of the availability of clean drinking water for human beings  is obvious to me. No degree of economic benefit should be allowed to endanger this availability of clean water."""
comment_3 = """Over the last years, perfluorocarbons and perfluoropolyethers have been frequently used as dielectric fluids for immersion cooling applications. The use of such liquids entails a high environmental impact as well as severe health risks during operation, maintenance and handling of the immersed equipment. Recently, new and more sustainable dielectric liquids – able to efficiently operate these systems - have entered the market, resulting in no need for per- and polyfluoroalkyl substances anymore. 9165 Cleaning and heat transfer: engineered fluids. Electronics: heat transfer fluid for (single-phase) immersion cooling (in EV, IT equipment). 9165       PFAS alternatives for (single-phase) immersion cooling applications. Recently, sustainable hydrocarbon (e.g. DC COOLING BioLife 4 from TotalEnergies) and ester (e.g. NatureCool2000 from Cargill, Mivolt DF7 from M&I Materials) based dielectric liquids have entered the immersion cooling market."""
comment_4 = """I think it is incredibly important that PFAS disappears and that the restriction proposal is therefore adopted without being watered down. As a citizen and consumer within the EU, one should be able to trust that the goods one purchases are free from carcinogenic and hormone-disrupting chemicals. As a citizen, I want to be able to trust that the EU has my health in focus, and adopting the restriction proposal would signal this. I am prepared for goods to have deteriorated properties as a result of PFAS disappearing, but I believe that the positive effects of discontinuing the use of PFAS are much greater than the negatives."""
comment_5 = """As part of the ZeroPM project ( Stockholm University is building a database of alternatives to persistent, mobile and toxic (PMT) substances, and to per- and polyfluoroalkyl substances (PFAS). We have attached a short report which presents the general structure of the database and the method we have followed to build it.  The beta-version of this database will be publicly available on the ZeroPM website ( ) from the 30th of September 2023, and will be continuously updated as we get more information.  We hope that this resource will support the development of the restriction on uses of PFAS, by providing an information repository of where researchers, innovators and business can consider alternatives to PFAS before the restriction targets. 9280       The aim of the database is to identify the functions provided by PFAS in their different uses and potential alternatives capable of providing similar functions.  We provide a summary of the number of functions provided by PFAS and the number of potential alternatives we have identified for each application in attached document. When relevant, we also provide a list of alternatives which we have identified but were not included in the restriction, to the best of our knowledge. For now, we have identified 520 alternatives to PFAS across all use categories evaluated (but excluding alternatives to PFAS used as active substance and as ingredients in cosmetic products). The database will be updated in the future. Among other, we aim to deepen the search of alternatives for specific use cases of PFAS in order to identify more potential alternatives."""

comment_6 = """MicrotracBEL supports the two statements made by FCJ and JFIA on the issues of proposed restriction, as per attached in section IV. 9269 PTFE thread sealing tape, semicondactors as compornents, other uses of FKM, PTFE and PCTFE as sealing materials for high vacuum. End use is in analytical instruments for powder and particle characterisation. 9269     a. FKM, PTFE, PCTFE. Components for semiconductors using FKM and PCTFE are also used, giving problems to the supply of this component. The anual tonnage and emissions No annual tonnage and emissions information is available. b. Used as tubing, sealing material and gaskets for high vacuum applications. Heat resistant, corrosion resistant, low moisture absorption and flexible. c. Our company produces analytical instruments for characterising powder and granular materials. These instruments are used in a wide range of industries, including catalysts, batteries, pharmaceuticals, and electronic materials for research and quality control purposes. There are no specific figures, but many companies use analytical instruments for powder analysis. d. There are no effective alternatives that can guarantee equivalent performance. e.f. No alternative candidates are available. g. No information is available."""
comment_7 = """As engineering service provider, planner and constructor for large-scale plant construction, we specify and handle various materials and material groups. Among others, the following semi-finished products and assemblies made of fluoropolymers are used in our range of activities: - Fluoropolymers as sealing materials (flat, O-ring, mechanical seal, etc.) in various equipment/pipelines - Fluoropolymers as lining materials in pumps and pipeline materials - Fluoropolymers as membrane materials in pumps, electrolyzers Additionally, fluoropolymers are indirectly used as coolants or additives in associated processes, which however fall within the scope of the end user/customer. As the broad field of mechanical and plant engineering is not listed in the sectors mentioned in Annex XV (Table 9) of the restriction report, the following areas were selected as the most relevant - Energy sector (Annex E.2.12.) - Sector as a whole - Petroleum and mining (Annex E.2.15.) - Fluoropolymer applications Furthermore, see SECTION III. Non-confidential comments. 9567" "We, as large-scale plant builders, can divide the percentage emission shares for our scope as follows: 0% emissions in the production phase (as we appear as end users) 0% emissions in the usage phase 100% emissions in the end-of-life phase (this is outside our scope and not considered) 9567"    "See SECTION III. Non-confidential comments"""

comment_8 = """The European Society for Medical Oncology (ESMO) notes the draft proposal’s aims to prevent PFAS accumulation in the environment and food chain and welcomes its efforts to improve human health. However, it is pivotal that such action is based on a comprehensive and consistent review of the available evidence as there are growing concerns about the possible impact of the draft proposal on the production, availability and manufacturing of cancer medicines. 9551 Pharmaceuticals, medical devices, excipients, tarting materials and chemical intermediates, reagents, solvents, catalysts, auxiliaries. """
comment_9 = """We would like to provide specific information on the use of PFAS in the chemical industry (chemicals synthesis). See two attached documents (one non-confidential version and one confidential version)."""


In [41]:
def few_imbalance_infavor(comment):
    return f"""
Stance detection is the task of identifying the explicit or implicit position expressed in a text with respect to a specific target.

You are a language model trained to analyze public consultation comments related to the proposed restriction of PFAS substances.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Stance 1: In favor

Example 2:
Comment 2: {comment_2}
Stance 2: In favor

Example 3:
Comment 3: {comment_3}
Stance 3: In favor

Example 4:
Comment 4: {comment_4}
Stance 4: In favor

Example 5:
Comment 5: {comment_5}
Stance 5: In favor

Example 6
Comment 6: {comment_6}
Stance 6: Against

Example 7:
Comment 7: {comment_7}
Stance 7: Against

Example 8:
Comment 8: {comment_8}
Stance 8: Neutral

Example 9:
Comment 9: {comment_9}
Stance 9: Neutral

Taking the examples into consideration, classify the stance expressed in the following comment toward the PFAS restriction proposal:

Comment:
\"{comment}\"

Possible labels:
- In favor
- Against
- Neutral

Answer with only one of the three labels.
"""

In [42]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["output_11.1"]:
        stance = get_response(few_imbalance_infavor(comment))
        df.at[idx, "output_11.1"] = stance
        print(f"[{idx}] Stance → {stance}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/90 [00:00<?, ?it/s]

[0] Stance → Against


Classifying comments:   1%|          | 1/90 [00:01<02:50,  1.91s/it]

Checkpoint saved at row 0
[1] Stance → In favor


Classifying comments:   2%|▏         | 2/90 [00:03<02:26,  1.67s/it]

...

[88] Stance → In favor


Classifying comments:  99%|█████████▉| 89/90 [02:26<00:01,  1.55s/it]

[89] Stance → Against


Classifying comments: 100%|██████████| 90/90 [02:28<00:00,  1.65s/it]

 Classification completed and saved.
